In [1]:
!git clone -b dev https://github.com/SebastianKrafft/DSAIT4030_DINO-WM.git dino_wm

Cloning into 'dino_wm'...
remote: Enumerating objects: 326, done.
remote: Counting objects: 100% (326/326), done.
remote: Compressing objects: 100% (218/218), done.
remote: Total 326 (delta 111), reused 300 (delta 91), pack-reused 0 (from 0)
Receiving objects: 100% (326/326), 5.58 MiB | 17.10 MiB/s, done.
Resolving deltas: 100% (111/111), done.


In [2]:
!pip install hydra-core einops dm_control dm_env pyyaml decord

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 2.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 MB 34.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 92.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 85.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 82.9 MB/s eta 0:00:00:00:01


In [11]:
import os
import wandb
Add-Ons
Secrets
Code Snippet

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB")

# Paste your actual key inside the quotes
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

True

In [10]:
import os

# IMPORTANT: Change this to the exact Kaggle path of your ORIGINAL PointMaze dataset!
os.environ['DATASET_DIR'] = '/kaggle/input/datasets/m1jktd/pnt-maze-emb-dinov3-part-1' 
print("Base data path set!")

Base data path set!


In [12]:
import os
import glob

# 1. Clean up
!rm -rf /kaggle/working/combined_embeddings
os.makedirs('/kaggle/working/combined_embeddings', exist_ok=True)

print("here")
# 2. Search literally everywhere in the input folder for .pt files
source_files = glob.glob('/kaggle/input/**/*.pt', recursive=True)

# Just in case you saved them as .pth instead of .pt!
if not source_files:
    source_files = glob.glob('/kaggle/input/**/*.pth', recursive=True)

if not source_files:
    print("ERROR: Still can't find them! Here is what Kaggle physically sees in your input folder:")
    # This will print every single file so we can see where they are hiding
    for root, dirs, files in os.walk('/kaggle/input/'):
        for name in files[:5]: # Print first 5 files per folder to avoid spam
            print(os.path.join(root, name))
else:
    print(f"Found {len(source_files)} embedding files! Building symlinks...")
    # 3. Create the symlinks
    for src in source_files:
        filename = os.path.basename(src)
        dst = os.path.join('/kaggle/working/combined_embeddings', filename)
        if not os.path.exists(dst):
            os.symlink(src, dst)
            
    # 4. Verify
    total_files = len(os.listdir('/kaggle/working/combined_embeddings'))
    print(f"Success! Merged {total_files} files into the virtual folder.")

here
Found 2000 embedding files! Building symlinks...
Success! Merged 2000 files into the virtual folder.


In [13]:
!du -sh /kaggle/working/combined_embeddings
!df -h /kaggle/working

7.9M	/kaggle/working/combined_embeddings
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   31M   20G   1% /kaggle/working


In [19]:
!find /kaggle/input -name "states.pth"

/kaggle/input/datasets/thegreenier/pointmaze/point_maze/states.pth


In [20]:
!cd dino_wm && HYDRA_FULL_ERROR=1 python train.py \
    --config-name train.yaml \
    env=point_maze \
    env.dataset.embedding_dir=/kaggle/working/combined_embeddings \
    env.dataset.data_path=/kaggle/input/datasets/thegreenier/pointmaze/point_maze \
    env.num_workers=0 \
    frameskip=5 \
    num_hist=3

[2026-06-03 15:38:35,151][__main__][INFO] - Model saved dir: /kaggle/working/dino_wm/outputs/2026-06-03/15-38-35
[2026-06-03 15:38:35,393][__main__][INFO] - rank: 0  model_name: 2026-06-03/15-38-35_point_maze_f5_h3_p1
[2026-06-03 15:38:35,394][__main__][INFO] - device: cuda   model_name: 2026-06-03/15-38-35_point_maze_f5_h3_p1
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: nmenta (nmenta-tu-delft) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ setting up run 1vq6019g (0.4s)
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/dino_wm/outputs/2026-06-03/15-38-35/wandb/run-20260603_153835-1vq6019g
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run quiet-fog-22
wandb: ⭐️ View project at https://wandb.ai/nmenta-tu-delft